In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [2]:
df=pd.read_csv("DataSet.csv")

In [3]:
df.head(3)

,Unnamed: 0,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F3915,F3916,F3917,F3918,F3919,F3920,F3921,F3922,F3923,F3924
0,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1,0,0,1,0,0,1,0,0
1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1,0,0,1,0,1,0,0,0
2,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1,0,0,1,0,0,1,0,0


##### The end columns are clearly customer/account profile data:

F3886 — account type: Savings, Current, MSME Micro, MSME Medium, Corp Adv,
F3888 — account open date (August 2011 entries),
F3890 — marital/status: R, SU, M, U,
F3891 — occupation: selfemployed, student, salaried, agriculture, housewife, retired,
F3892 — gender: M / F,
F3893 — segment: RETAIL / CORPORATE,
F3894 — age,

Likely label columns (binary 0/1 at the very end):

F3897, F3900–F3924 — multiple binary flags, likely fraud labels, alert flags, mule flags,

In [4]:
# target var
print(df['F3924'].value_counts())

# Given important features
features = ['F115','F321','F527','F531','F670','F1692','F2082',
            'F2122','F2582','F2678','F2737','F2956','F3043',
            'F3836','F3887','F3889','F3891','F3894','F3924']

df_model = df[features].copy()
print(df_model.shape)
print(df_model.isnull().sum())
print(df_model.dtypes)

F3924
0    9001
1      81
Name: count, dtype: int64
(9082, 19)
F115      358
F321       85
F527      788
F531      641
F670       16
F1692       7
F2082       7
F2122       4
F2582    3404
F2678    2574
F2737     112
F2956    1025
F3043    5824
F3836       0
F3887       0
F3889       0
F3891       0
F3894       3
F3924       0
dtype: int64
F115     float64
F321     float64
F527     float64
F531     float64
F670     float64
F1692    float64
F2082    float64
F2122    float64
F2582    float64
F2678    float64
F2737    float64
F2956    float64
F3043    float64
F3836    float64
F3887      int64
F3889     object
F3891     object
F3894    float64
F3924      int64
dtype: object


In [5]:
df_model.head()

,F115,F321,F527,F531,F670,F1692,F2082,F2122,F2582,F2678,F2737,F2956,F3043,F3836,F3887,F3889,F3891,F3894,F3924
0,0.53,1.09,1.32,1.12,1.0,1.0,0.0,0.009091,-0.08,NaN,-0.16,36.0,NaN,29814.53,170,G365D,selfemployed,30.0,0
1,0.30,1.19,0.77,0.76,0.0,0.0,0.0,0.000000,-0.25,-0.47,-0.05,99.0,NaN,371320.98,169,G365D,selfemployed,32.0,0
2,0.43,1.25,1.37,1.68,0.0,0.0,0.0,0.000000,-0.21,-0.46,0.00,100.0,NaN,546659.78,170,G365D,student,32.0,0
3,0.69,2.02,0.92,1.92,0.0,0.0,0.0,0.000000,0.09,-0.46,0.66,56.0,114.0,353339.72,170,G365D,selfemployed,45.0,0
4,0.53,0.78,1.23,0.74,0.0,0.0,0.0,0.000000,0.65,-0.42,-0.53,63.0,109.0,1116.11,171,G365D,student,30.0,0


# EDA

In [6]:
df_model.drop(columns=['F3043'], inplace=True)

In [7]:
num_cols = ['F115','F321','F527','F531','F670','F1692','F2082',
            'F2122','F2582','F2678','F2737','F2956','F3836','F3894']
for col in num_cols:
    df_model[col].fillna(df_model[col].median(), inplace=True)

In [8]:
# encoding str clms
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df_model['F3889'] = le.fit_transform(df_model['F3889'].astype(str))
df_model['F3891'] = le.fit_transform(df_model['F3891'].astype(str))

In [9]:
print(df_model.isnull().sum())
print(df_model.shape)

F115     0
F321     0
F527     0
F531     0
F670     0
F1692    0
F2082    0
F2122    0
F2582    0
F2678    0
F2737    0
F2956    0
F3836    0
F3887    0
F3889    0
F3891    0
F3894    0
F3924    0
dtype: int64
(9082, 18)


In [10]:
# pip install imbalanced-learn



In [19]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import joblib, json

X = df_model.drop(columns=['F3924'])
y = df_model['F3924']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SMOTE
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)
print("After SMOTE:", y_train_sm.value_counts())

# Train XGBoost
model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=1,
    random_state=42,
    eval_metric='logloss'
)
model.fit(X_train_sm, y_train_sm)

After SMOTE: F3924
0    7200
1    7200
Name: count, dtype: int64


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=200, n_jobs=None,
              num_parallel_tree=None, ...)

In [18]:
# Predict with threshold 0.15
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.15).astype(int)

# Evaluate
print(classification_report(y_test, y_pred))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

              precision    recall  f1-score   support

           0       1.00      0.80      0.89      1801
           1       0.04      0.94      0.08        16

    accuracy                           0.80      1817
   macro avg       0.52      0.87      0.48      1817
weighted avg       0.99      0.80      0.88      1817

ROC-AUC: 0.9065


In [20]:
# Save
joblib.dump(model, 'fraud_model.pkl')
model_meta = {
    "features": list(X.columns),
    "threshold": 0.15
}
with open('model_meta.json', 'w') as f:
    json.dump(model_meta, f)

print("\nSaved fraud_model.pkl and model_meta.json")
print("Features:", list(X.columns))


Saved fraud_model.pkl and model_meta.json
Features: ['F115', 'F321', 'F527', 'F531', 'F670', 'F1692', 'F2082', 'F2122', 'F2582', 'F2678', 'F2737', 'F2956', 'F3836', 'F3887', 'F3889', 'F3891', 'F3894']
